<a href="https://colab.research.google.com/github/AuliaRestyy/machineLearning/blob/js09/JS09_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kagglehub annoy faiss-cpu hnswlib scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 7.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 14.1 MB/s  0:00:01
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=521287 sha256=b76f915961129f35d7d992ef484eef0f43c65d29dfd3f096bc7e6866d498609f
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2683572 sha256=00812a1b82a03f765a8487745985b681cdc72e8b195a372837b85d57d0d59f37
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [2]:
import pandas as pd
import numpy as np
import time
import os
import kagglehub
import faiss
import hnswlib

from annoy import AnnoyIndex
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")
print("Dataset path:", path)

Using Colab cache for faster access to the 'spotify-songs-with-attributes-and-lyrics' dataset.
Dataset path: /kaggle/input/spotify-songs-with-attributes-and-lyrics


In [3]:
file_path = None

for root, dirs, files in os.walk(path):
    for file in files:
        if file == 'songs_with_attributes_and_lyrics.csv':
            file_path = os.path.join(root, file)

print("Selected file:", file_path)

features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo'
]

columns = ['id', 'name', 'artists'] + features
df = pd.read_csv(file_path, usecols=columns)

print("Dataset shape:", df.shape)
print(df.head())

Selected file: /kaggle/input/spotify-songs-with-attributes-and-lyrics/songs_with_attributes_and_lyrics.csv
Dataset shape: (955320, 12)
                       id             name   artists  danceability  energy  \
0  0Prct5TDjAnEgIqbxcldY9                !  HELLYEAH         0.415  0.6050   
1  2ASl4wirkeYm3OWZxXKYuq               !!   Yxngxr1         0.788  0.6480   
2  69lcggVPmOr9cvPx9kLiiN  !!! - Interlude    Glowie         0.000  0.0354   
3  4U7dlZjg1s9pjdppqZy0fm   !!De Repente!!   Rosendo         0.657  0.8820   
4  4v1IBp3Y3rpkWmWzIlkYju   !!De Repente!!   Rosendo         0.659  0.8930   

   loudness  speechiness  acousticness  instrumentalness  liveness  valence  \
0   -11.157       0.0575       0.00116          0.838000    0.4710    0.193   
1    -9.135       0.3150       0.90000          0.000000    0.1760    0.287   
2   -20.151       0.0000       0.90800          0.000000    0.4790    0.000   
3    -6.340       0.0385       0.00740          0.000013    0.0474    0.939   
4

In [4]:
print("Missing values:")
print(df[features].isnull().sum())

df = df.dropna(subset=features).reset_index(drop=True)
X = df[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = X_scaled.astype('float32')

print("Scaled dataset shape:", X_scaled.shape)

Missing values:
danceability        0
energy              0
loudness            0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
dtype: int64
Scaled dataset shape: (955320, 9)


In [5]:
k = 10

query_index = 0
query = X_scaled[query_index].reshape(1, -1)

print("Query Track:")
print("Name:", df.iloc[query_index]['name'])
print("Artist:", df.iloc[query_index]['artists'])

Query Track:
Name: !
Artist: HELLYEAH


In [6]:
nn = NearestNeighbors(
    n_neighbors=k + 1,
    algorithm='brute',
    metric='euclidean'
)

nn.fit(X_scaled)
start = time.time()
distances_exact, indices_exact = nn.kneighbors(query)
time_exact = time.time() - start
indices_exact = indices_exact[0][1:]
distances_exact = distances_exact[0][1:]

print("=== Exact NN ===")
print("Indices:", indices_exact)
print("Distances:", distances_exact)
print("Query Time:", time_exact, "seconds")

=== Exact NN ===
Indices: [394553 764272 837727 749223  61511 625227 833164 676733 523698  85956]
Distances: [0.88665658 0.90944272 0.93600774 0.93759984 0.94650489 0.97633171
 0.99541754 1.02076137 1.04078293 1.04566109]
Query Time: 0.06253814697265625 seconds


In [7]:
dim = X_scaled.shape[1]
ann_index = AnnoyIndex(dim, 'euclidean')
start = time.time()

for i, vec in enumerate(X_scaled):
    ann_index.add_item(i, vec)

ann_index.build(10)
build_time_annoy = time.time() - start
start = time.time()

indices_annoy, distances_annoy = ann_index.get_nns_by_item(
    query_index,
    k + 1,
    include_distances=True
)

time_annoy = time.time() - start
indices_annoy = indices_annoy[1:]
distances_annoy = distances_annoy[1:]

print("=== ANNOY ===")
print("Indices:", indices_annoy)
print("Distances:", distances_annoy)
print("Build Time:", build_time_annoy, "seconds")
print("Query Time:", time_annoy, "seconds")

=== ANNOY ===
Indices: [764272, 837727, 749223, 61511, 833164, 676733, 523698, 763625, 256217, 423267]
Distances: [0.909442663192749, 0.9360076785087585, 0.9375998377799988, 0.9465048909187317, 0.9954175353050232, 1.0207613706588745, 1.0407829284667969, 1.0519623756408691, 1.0535550117492676, 1.0942964553833008]
Build Time: 17.29822087287903 seconds
Query Time: 0.00021004676818847656 seconds


In [8]:
nlist = 10
m = 3
quantizer = faiss.IndexFlatL2(dim)

faiss_index = faiss.IndexIVFPQ(
    quantizer,
    dim,
    nlist,
    m,
    8
)

start = time.time()
faiss_index.train(X_scaled)
faiss_index.add(X_scaled)
build_time_faiss = time.time() - start
start = time.time()

distances_faiss, indices_faiss = faiss_index.search(query, k + 1)

time_faiss = time.time() - start
indices_faiss = indices_faiss[0][1:]
distances_faiss = distances_faiss[0][1:]

print("=== FAISS ===")
print("Indices:", indices_faiss)
print("Distances:", distances_faiss)
print("Build Time:", build_time_faiss, "seconds")
print("Query Time:", time_faiss, "seconds")

=== FAISS ===
Indices: [763625 837727  61511 523698 749223 254260 526123 394553 833164 385544]
Distances: [0.38326693 0.72585535 0.7746291  0.77772427 0.81193686 0.84322214
 0.8488498  0.8869643  0.89664125 0.9137492 ]
Build Time: 8.623927593231201 seconds
Query Time: 0.0010313987731933594 seconds


In [9]:
hnsw_index = hnswlib.Index(
    space='l2',
    dim=dim
)

start = time.time()

hnsw_index.init_index(
    max_elements=len(X_scaled),
    ef_construction=200,
    M=16
)

hnsw_index.add_items(X_scaled)
hnsw_index.set_ef(50)
build_time_hnsw = time.time() - start
start = time.time()

labels_hnsw, distances_hnsw = hnsw_index.knn_query(query, k=k + 1)

time_hnsw = time.time() - start
indices_hnsw = labels_hnsw[0][1:]
distances_hnsw = distances_hnsw[0][1:]

print("=== HNSW ===")
print("Indices:", indices_hnsw)
print("Distances:", distances_hnsw)
print("Build Time:", build_time_hnsw, "seconds")
print("Query Time:", time_hnsw, "seconds")

=== HNSW ===
Indices: [394553 764272 837727 749223  61511 625227 833164 676733 523698  85956]
Distances: [0.7861599  0.82708603 0.8761104  0.87909347 0.8958715  0.9532235
 0.9908561  1.0419539  1.0832292  1.0934073 ]
Build Time: 148.2101833820343 seconds
Query Time: 0.0003399848937988281 seconds


In [10]:
print("=== QUERY TRACK ===")
print("Name:", df.iloc[query_index]['name'])
print("Artist:", df.iloc[query_index]['artists'])

print("\n=== EXACT NN RESULTS ===")
for i in indices_exact:
    print(df.iloc[i]['name'], "-", df.iloc[i]['artists'])

print("\n=== ANNOY RESULTS ===")
for i in indices_annoy:
    print(df.iloc[i]['name'], "-", df.iloc[i]['artists'])

print("\n=== FAISS RESULTS ===")
for i in indices_faiss:
    print(df.iloc[i]['name'], "-", df.iloc[i]['artists'])

print("\n=== HNSW RESULTS ===")
for i in indices_hnsw:
    print(df.iloc[i]['name'], "-", df.iloc[i]['artists'])

=== QUERY TRACK ===
Name: !
Artist: HELLYEAH

=== EXACT NN RESULTS ===
Intro (Who The Fuck Are You') - C-Lekktor
Testimonium - Lacrimosa
Ultimate Satisfaction - The Field Remix - Bear In Heaven
Sæglópur - Sigur Rós
Autumn Interlude - Amethystium
Púrpura - Margaritas Podridas
Turn from Heaven - Therion
See You When I Die - BOYO
Mrityu - Multani;Karun;Udbhav
Bewitched - The Wedding Present

=== ANNOY RESULTS ===
Testimonium - Lacrimosa
Ultimate Satisfaction - The Field Remix - Bear In Heaven
Sæglópur - Sigur Rós
Autumn Interlude - Amethystium
Turn from Heaven - Therion
See You When I Die - BOYO
Mrityu - Multani;Karun;Udbhav
Teri Yaadon Se - Lofi Flip - Jammy Weirdo;Mustafa Zahid;Pranay.M. Rijia
Feed the Fire - Spiral Drive
Kindred Soul - Mothboy Remix - Stray, Unter Null

=== FAISS RESULTS ===
Teri Yaadon Se - Lofi Flip - Jammy Weirdo;Mustafa Zahid;Pranay.M. Rijia
Ultimate Satisfaction - The Field Remix - Bear In Heaven
Autumn Interlude - Amethystium
Mrityu - Multani;Karun;Udbhav
Sæglópu

In [11]:
results = pd.DataFrame({
    'Algorithm': [
        'Exact NN',
        'ANNOY',
        'FAISS',
        'HNSW'
    ],

    'Build Time (seconds)': [
        '-',
        build_time_annoy,
        build_time_faiss,
        build_time_hnsw
    ],

    'Query Time (seconds)': [
        time_exact,
        time_annoy,
        time_faiss,
        time_hnsw
    ]
})

display(results)

,Algorithm,Build Time (seconds),Query Time (seconds)
0,Exact NN,-,0.062538
1,ANNOY,17.298221,0.000210
2,FAISS,8.623928,0.001031
3,HNSW,148.210183,0.000340


Based on the experiment results, each algorithm has different performance when searching for similar songs based on audio features. FAISS has the shortest build time at 8.62 seconds, while HNSW takes the longest at 148.21 seconds. For query time, ANNOY is the fastest at 0.000210 seconds, followed by HNSW at 0.000340 seconds and FAISS at 0.001031 seconds. Meanwhile, Exact NN takes 0.062538 seconds. These results show that Approximate Nearest Neighbors (ANN) algorithms can perform faster searches than Exact NN in this experiment. However, each algorithm has different build times and search speeds.